####01. Driver Standings

In [0]:
%sql
CREATE OR REPLACE TABLE f1_catalog.gold.driver_standings AS

SELECT d.driver_id, d.driver_name,
COUNT(DISTINCT r.race_id) AS races_participated,
SUM(r.points) AS total_points,
COUNT(CASE
WHEN TRY_CAST(r.position AS INT) = 1 THEN 1
END) AS total_wins,
COUNT(CASE
WHEN TRY_CAST(r.position AS INT) BETWEEN 1 AND 3 THEN 1
END) AS total_podiums,
ROUND(SUM(r.points) / NULLIF(COUNT(DISTINCT r.race_id), 0), 2) AS avg_points_per_race,
ROUND(COUNT(CASE WHEN TRY_CAST(r.position AS INT) = 1 THEN 1 END)
* 100.0 / NULLIF(COUNT(DISTINCT r.race_id), 0), 2) AS win_percentage
FROM f1_catalog.silver.results r
INNER JOIN f1_catalog.silver.drivers d ON r.driver_id = d.driver_id
GROUP BY d.driver_id, d.driver_name;

In [0]:
%sql
SELECT *
FROM f1_catalog.gold.driver_standings
ORDER BY total_points DESC
LIMIT 20;

##### Driver Standings

Evaluates overall driver performance using total points, race wins, podium finishes, races participated, average points per race and win percentage.

####02. Constructor Standings

In [0]:
%sql
CREATE OR REPLACE TABLE f1_catalog.gold.constructor_standings AS

SELECT c.constructor_id, c.name AS constructor_name,
COUNT(DISTINCT r.race_id) AS races_participated,
SUM(r.points) AS total_points,
COUNT(CASE
WHEN TRY_CAST(r.position AS INT) = 1 THEN 1
END) AS total_wins,
COUNT(CASE
WHEN TRY_CAST(r.position AS INT) BETWEEN 1 AND 3 THEN 1
END) AS total_podium_finishes,
ROUND(SUM(r.points) / NULLIF(COUNT(DISTINCT r.race_id), 0), 2) AS avg_points_per_race,
ROUND(COUNT(CASE
WHEN TRY_CAST(r.position AS INT) = 1 THEN 1
END) * 100.0 / NULLIF(COUNT(DISTINCT r.race_id), 0), 2) AS win_percentage
FROM f1_catalog.silver.results r
INNER JOIN f1_catalog.silver.constructor c ON r.constructor_id = c.constructor_id
GROUP BY c.constructor_id, c.name;

In [0]:
%sql
SELECT *
FROM f1_catalog.gold.constructor_standings
ORDER BY total_points DESC
LIMIT 20;

#####Constructor Standings

Compares constructor performance based on total points, race wins, podium finishes, races participated and average points per race.

####03. Driver's Yearly Performance

In [0]:
%sql

CREATE OR REPLACE TABLE f1_catalog.gold.driver_yearly_performance AS

WITH driver_performance AS (SELECT d.driver_id, d.driver_name, ra.year AS season,
COUNT(DISTINCT r.race_id) AS races_participated,
SUM(r.points) AS total_points,
COUNT(CASE
WHEN TRY_CAST(r.position AS INT) = 1 THEN 1
END) AS total_wins,
COUNT(CASE
WHEN TRY_CAST(r.position AS INT) BETWEEN 1 AND 3 THEN 1
END) AS total_podiums
FROM f1_catalog.silver.results r
INNER JOIN f1_catalog.silver.drivers d ON r.driver_id = d.driver_id
INNER JOIN f1_catalog.silver.races ra ON r.race_id = ra.race_id
GROUP BY d.driver_id, d.driver_name, ra.year)

SELECT *, ROUND(total_points / NULLIF(races_participated, 0), 2) AS avg_points_per_race,
DENSE_RANK() OVER (PARTITION BY season ORDER BY total_points DESC) AS championship_rank
FROM driver_performance;

In [0]:
%sql

SELECT *
FROM f1_catalog.gold.driver_yearly_performance
WHERE season = 2023
ORDER BY championship_rank;

##### Driver Yearly Performance

Analyzes individual driver performance across seasons using yearly points, wins, podiums, races participated and championship ranking.

####04. Driver wins count

In [0]:
%sql

CREATE OR REPLACE TABLE f1_catalog.gold.driver_wins AS

SELECT d.driver_id, d.driver_name,
COUNT(*) AS total_wins,
COUNT(DISTINCT ra.year) AS winning_seasons,
MIN(ra.year) AS first_winning_season,
MAX(ra.year) AS latest_winning_season,
SUM(r.points) AS points_from_wins
FROM f1_catalog.silver.results r
INNER JOIN f1_catalog.silver.drivers d ON r.driver_id = d.driver_id
INNER JOIN f1_catalog.silver.races ra ON r.race_id = ra.race_id
WHERE TRY_CAST(r.position AS INT) = 1
GROUP BY d.driver_id, d.driver_name;

In [0]:
%sql

SELECT *
FROM f1_catalog.gold.driver_wins
ORDER BY total_wins DESC;

#####Driver Wins

Identifies successful drivers by analyzing total race victories, winning seasons, first and last winning seasons, and points earned from victories.

####05. Fastest Lap 

In [0]:
%sql

CREATE OR REPLACE TABLE f1_catalog.gold.fastest_laps AS

WITH lap_times AS (
SELECT d.driver_id, d.driver_name, r.fastestLapTime,
(CAST(SPLIT(r.fastestLapTime, ':')[0] AS DOUBLE) * 60000
+ CAST(SPLIT(r.fastestLapTime, ':')[1] AS DOUBLE) * 1000) AS lap_time_ms
FROM f1_catalog.silver.results r
INNER JOIN f1_catalog.silver.drivers d ON r.driver_id = d.driver_id
WHERE r.fastestLapTime IS NOT NULL
AND r.fastestLapTime RLIKE '^[0-9]+:[0-9]+\\.[0-9]+$'),
ranked_laps AS (
SELECT *,
ROW_NUMBER() OVER (PARTITION BY driver_id ORDER BY lap_time_ms ASC) AS lap_rank
FROM lap_times)

SELECT driver_id, driver_name, fastestLapTime AS best_lap_time,
ROUND(lap_time_ms, 0) AS best_lap_time_ms
FROM ranked_laps
WHERE lap_rank = 1;

In [0]:
%sql

SELECT *
FROM f1_catalog.gold.fastest_laps
ORDER BY best_lap_time_ms ASC
LIMIT 20;

#####Fastest Laps

Identifies each driver's best recorded lap time and compares their fastest-lap performance.

####06. Pit Stop Analysis

In [0]:
%sql

CREATE OR REPLACE TABLE f1_catalog.gold.pitstop_analysis AS

SELECT d.driver_id, d.driver_name, c.constructor_id, c.name AS constructor_name, 
ra.year AS season,
COUNT(*) AS total_pitstops,
COUNT(DISTINCT p.race_id) AS races_with_pitstops,
ROUND(AVG(p.milliseconds) / 1000, 2) AS avg_pitstop_seconds,
ROUND(MIN(p.milliseconds) / 1000, 2) AS fastest_pitstop_seconds,
ROUND(MAX(p.milliseconds) / 1000, 2) AS slowest_pitstop_seconds
FROM f1_catalog.silver.pitstops p
INNER JOIN f1_catalog.silver.drivers d ON p.driver_id = d.driver_id
INNER JOIN f1_catalog.silver.races ra ON p.race_id = ra.race_id
INNER JOIN f1_catalog.silver.results r ON p.race_id = r.race_id
AND p.driver_id = r.driver_id
INNER JOIN f1_catalog.silver.constructor c ON r.constructor_id = c.constructor_id
WHERE p.milliseconds IS NOT NULL
GROUP BY d.driver_id, d.driver_name, c.constructor_id, c.name, ra.year;

In [0]:
%sql

SELECT *
FROM f1_catalog.gold.pitstop_analysis
ORDER BY avg_pitstop_seconds ASC
LIMIT 20;

#####Pitstop Analysis

Evaluates pitstop efficiency across drivers, constructors and seasons using total pitstops, average duration, fastest duration and slowest duration.

####07. Race Results

In [0]:
%sql

CREATE OR REPLACE TABLE f1_catalog.gold.race_results AS

SELECT ra.race_id, ra.name AS race_name, ra.year AS season,
d.driver_id, d.driver_name,
c.constructor_id, c.name AS constructor_name,
r.grid AS starting_position,
TRY_CAST(r.position AS INT) AS finishing_position,
    CASE
        WHEN TRY_CAST(r.position AS INT) IS NOT NULL
        THEN r.grid - TRY_CAST(r.position AS INT)
        ELSE NULL END AS positions_gained, 
        r.points, r.laps AS laps_completed,
    CASE
        WHEN TRY_CAST(r.position AS INT) = 1 THEN 'Winner'
        WHEN TRY_CAST(r.position AS INT) BETWEEN 2 AND 3 THEN 'Podium'
        WHEN TRY_CAST(r.position AS INT) IS NOT NULL THEN 'Finished'
        ELSE 'Not Classified'
    END AS race_performance
FROM f1_catalog.silver.results r
INNER JOIN f1_catalog.silver.races ra ON r.race_id = ra.race_id
INNER JOIN f1_catalog.silver.drivers d ON r.driver_id = d.driver_id
INNER JOIN f1_catalog.silver.constructor c ON r.constructor_id = c.constructor_id;

In [0]:
%sql

SELECT *
FROM f1_catalog.gold.race_results
ORDER BY season DESC, race_id
LIMIT 20;

#####Race Results

Provides driver-level race performance, including starting position, finishing position, positions gained, points scored, laps completed and race classification.

#### 08. Constructor Yearly Performance


In [0]:
%sql

CREATE OR REPLACE TABLE f1_catalog.gold.constructor_yearly_performance AS

WITH constructor_performance AS (
SELECT c.constructor_id, c.name AS constructor_name, ra.year AS season,
COUNT(DISTINCT r.race_id) AS races_participated,
SUM(r.points) AS total_points,
COUNT(
CASE
    WHEN TRY_CAST(r.position AS INT) = 1 THEN 1
END) AS total_wins,
COUNT(
CASE
    WHEN TRY_CAST(r.position AS INT) BETWEEN 1 AND 3 THEN 1
END) AS total_podium_finishes
FROM f1_catalog.silver.results r
INNER JOIN f1_catalog.silver.constructor c ON r.constructor_id = c.constructor_id
INNER JOIN f1_catalog.silver.races ra ON r.race_id = ra.race_id
GROUP BY c.constructor_id, c.name, ra.year)

SELECT *,
ROUND(total_points / NULLIF(races_participated, 0), 2) AS avg_points_per_race,
DENSE_RANK() OVER ( PARTITION BY season ORDER BY total_points DESC) 
AS constructor_championship_rank
FROM constructor_performance;

In [0]:
%sql

SELECT *
FROM f1_catalog.gold.constructor_yearly_performance
WHERE season = 2020
ORDER BY constructor_championship_rank;

#####Constructor Yearly Performance

Tracks constructor performance across seasons using yearly points, race wins, podium finishes, races participated and championship ranking.

#### 09. Season Summary

In [0]:
%sql

CREATE OR REPLACE TABLE f1_catalog.gold.season_summary AS

WITH driver_stats AS (
SELECT ra.year AS season, d.driver_name,
SUM(r.points) AS total_points,
COUNT(CASE WHEN TRY_CAST(r.position AS INT) = 1 THEN 1 END) AS total_wins,
DENSE_RANK() OVER (PARTITION BY ra.year ORDER BY SUM(r.points) DESC ) AS driver_rank
FROM f1_catalog.silver.results r 
JOIN f1_catalog.silver.races ra ON r.race_id = ra.race_id
JOIN f1_catalog.silver.drivers d ON r.driver_id = d.driver_id
GROUP BY ra.year, d.driver_name),

constructor_stats AS (
SELECT ra.year AS season, c.name AS constructor_name,
SUM(r.points) AS total_points,
DENSE_RANK() OVER (PARTITION BY ra.year ORDER BY SUM(r.points) DESC) AS constructor_rank
FROM f1_catalog.silver.results r
JOIN f1_catalog.silver.races ra ON r.race_id = ra.race_id
JOIN f1_catalog.silver.constructor c ON r.constructor_id = c.constructor_id
GROUP BY ra.year, c.name),

season_stats AS (
SELECT ra.year AS season,
COUNT(DISTINCT ra.race_id) AS total_races,
COUNT(DISTINCT r.driver_id) AS total_drivers,
COUNT(DISTINCT r.constructor_id) AS total_constructors,
SUM(r.points) AS total_points
FROM f1_catalog.silver.races ra
JOIN f1_catalog.silver.results r ON ra.race_id = r.race_id
GROUP BY ra.year)

SELECT s.season, s.total_races,s.total_drivers, s.total_constructors,
s.total_points, d.driver_name AS leading_driver,
d.total_points AS leading_driver_points,
c.constructor_name AS leading_constructor,
c.total_points AS leading_constructor_points
FROM season_stats s
LEFT JOIN driver_stats d ON s.season = d.season AND d.driver_rank = 1
LEFT JOIN constructor_stats c ON s.season = c.season AND c.constructor_rank = 1;

In [0]:
%sql

SELECT *
FROM f1_catalog.gold.season_summary
ORDER BY season DESC;

#####Season Summary

Provides a season-level overview using total races, participating drivers, constructors, points scored, leading driver and leading constructor.

#### 10. Race Summary

In [0]:
%sql

CREATE OR REPLACE TABLE f1_catalog.gold.race_summary AS

SELECT ra.race_id, ra.name AS race_name, ra.year AS season,
COUNT(DISTINCT r.driver_id) AS total_drivers,
COUNT(DISTINCT r.constructor_id) AS total_constructors,
MAX(
CASE
WHEN TRY_CAST(r.position AS INT) = 1 THEN d.driver_name
END) AS race_winner,
MAX(
CASE
WHEN TRY_CAST(r.position AS INT) = 1 THEN c.name
END) AS winning_constructor,
SUM(r.points) AS total_points_awarded,
ROUND(AVG(TRY_CAST(r.position AS DOUBLE)), 2) AS avg_finishing_position,
SUM(r.laps) AS total_laps_completed
FROM f1_catalog.silver.races ra
INNER JOIN f1_catalog.silver.results r ON ra.race_id = r.race_id
INNER JOIN f1_catalog.silver.drivers d ON r.driver_id = d.driver_id
INNER JOIN f1_catalog.silver.constructor c ON r.constructor_id = c.constructor_id
GROUP BY ra.race_id, ra.name, ra.year;

In [0]:
%sql

SELECT *
FROM f1_catalog.gold.race_summary
ORDER BY season DESC, race_id DESC
LIMIT 20;

#####Race Summary

Provides a race-level overview covering participating drivers and constructors, race winner, winning constructor, total points, average finishing position and completed laps.

### Gold Layer – Overall Conclusion

The Gold Layer transforms cleaned Silver data into business-ready analytical datasets. It contains 10 analytical tables covering driver performance, constructor performance, race results, pitstop efficiency and seasonal trends.

These datasets provide meaningful KPIs for evaluating historical Formula 1 performance and serve as the foundation for data visualization and business insights.
